# Random Forest Classifier Implementation Notebook

This notebook contains the RFC model building and testing on the three dataset. 

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, ndcg_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score
from sklearn.metrics import average_precision_score

# Load datasets

In [ ]:
# Load feature datasets
X_train_roberta = pd.read_csv('X_train_roberta.csv')
X_test_roberta = pd.read_csv('X_test_roberta.csv')

X_train_distilroberta = pd.read_csv('X_train_distilroberta.csv')
X_test_distilroberta = pd.read_csv('X_test_distilroberta.csv')

X_train_distilroberta_mitch = pd.read_csv('X_train_distilroberta_mitch.csv')
X_test_distilroberta_mitch = pd.read_csv('X_test_distilroberta_mitch.csv')

# Load target datasets
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

target_group_train = pd.read_csv('target_train.csv')
target_group_test = pd.read_csv('target_test.csv')

# Preprocess Data

## Feature Engineering
The release_year column was grouped into generation because realse year had 56 unique values which can cause high dimensionality.

## Categorical feature encoding
In the cell below, the predictor categorical variable was encoded to numerical value [0,1] using one-hot encoding. One-hot encoding is used so that the model will not misinterpret the encoded value as a hierarchy or order. The target variable was encoded using label encoder because it is binary.

## Numerical feature Normalisation

The numerical value was normalised because the box plot in Exploratory Data Analysis Notebook showed that the values had diverse range.

In [ ]:
def extract_year_from_date(date_str):
    """
    Extract the year from a date string using regex to find the first 4-digit sequence.
    """
    match = re.search(r'\d{4}', date_str)
    if match:
        return int(match.group(0))  # Return the first 4-digit match as an integer
    return None  # Return None if no match is found

def map_year_to_generation(year):
    """
    Map a given year to a generation name.
    """
    if year >= 2013:
        return 'Generation Alpha'
    elif year >= 1997:
        return 'Generation Z'
    elif year >= 1981:
        return 'Millennials'
    elif year >= 1965:
        return 'Generation X'
    elif year >= 1946:
        return 'Baby Boomers'
    elif year >= 1928:
        return 'Silent Generation'
    else:
        return 'Unknown'

def preprocess_data(df, mood_column='final_mood_roberta', text_column = 'text_roberta'):
    # Extract the year using regex
    #df['year'] = df['release_date'].apply(extract_year_from_date)
    
    # Map the year to generation names
    df['generation'] = df['release_date'].apply(map_year_to_generation)
    
    # Drop unnecessary columns including 'year'
    df = df.drop(columns=['artist_name', text_column, 'release_date', 'first_genre', 'year'])
    
    # Standardize numerical features
    numerical_features = [
        'liveness', 'loudness', 'popularity', 'acousticness',
        'speechiness', 'tempo', 'valence', 'danceability', 'energy'
    ]
    
    scaler = MinMaxScaler()
    df[numerical_features] = scaler.fit_transform(df[numerical_features])
    
    # One-hot encode categorical features including the mood column
    categorical_features = [mood_column, 'target_group', 'generation']
    df = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)
    
    return df

In [ ]:
X_train_roberta= preprocess_data(X_train_roberta, mood_column = 'final_mood_roberta', text_column = 'text_roberta')
X_test_roberta= preprocess_data(X_test_roberta, mood_column = 'final_mood_roberta', text_column = 'text_roberta')

In [ ]:
X_train_distilroberta= preprocess_data(X_train_distilroberta, mood_column = 'final_mood_jhart', text_column = 'text_distilroberta')
X_test_distilroberta= preprocess_data(X_test_distilroberta, mood_column = 'final_mood_jhart', text_column = 'text_distilroberta')

In [ ]:
X_train_distilroberta_mitch= preprocess_data(X_train_distilroberta_mitch, mood_column = 'final_mood_mitch', text_column = 'text_distilroberta_mitch')
X_test_distilroberta_mitch= preprocess_data(X_test_distilroberta_mitch, mood_column = 'final_mood_mitch', text_column = 'text_distilroberta_mitch')

## Split Training Data to get 20% validation data

In [ ]:
from sklearn.model_selection import train_test_split

# Generate indices for the train-validation split
train_indices, val_indices = train_test_split(
    range(len(X_train_roberta)), test_size=0.2, random_state=42
)

# Use these indices to index each dataset
X_train_roberta_train = X_train_roberta.iloc[train_indices]
X_val_roberta = X_train_roberta.iloc[val_indices]

X_train_distilroberta_train = X_train_distilroberta.iloc[train_indices]
X_val_distilroberta = X_train_distilroberta.iloc[val_indices]

X_train_distilroberta_mitch_train = X_train_distilroberta_mitch.iloc[train_indices]
X_val_distilroberta_mitch = X_train_distilroberta_mitch.iloc[val_indices]

y_train_train = y_train.iloc[train_indices]
y_val = y_train.iloc[val_indices]

target_group_train_split = target_group_train.iloc[train_indices]
target_group_val_split = target_group_train.iloc[val_indices]


# Evaluation Metrics

In [ ]:
def calculate_precision_at_k(y_true, y_pred_proba, k=20):
  """Calculates precision@k for binary classification.

  Args:
    y_true: True labels.
    y_pred_proba: Predicted probabilities.
    k: Number of top predictions to consider.

  Returns:
    Precision@k value.
  """

  # Rank predictions by probability
  indices = np.argsort(y_pred_proba)[::-1]

  # Apply threshold and select top k
  y_pred = (y_pred_proba[indices] > 0.5).astype(int)
  y_pred_top_k = y_pred[:k]

  # Retrieve actual labels
  y_true_top_k = y_true[indices][:k]

  # Calculate numerator and denominator
  numerator = np.sum(y_pred_top_k * y_true_top_k)
  denominator = np.sum(y_pred_top_k)

  # Calculate precision@k
  if denominator > 0:
    precision_at_k = numerator / denominator
  else:
    precision_at_k = 0

  return precision_at_k

In [ ]:
def average_precision_top_k(y_true, y_pred_proba, k=20):
  """Calculates average precision for the top k predictions.

  Args:
    y_true: True binary labels.
    y_score: Target scores.
    k: Number of top predictions to consider.

  Returns:
    Average precision value.
  """

  # Filter top k predictions
  indices = np.argsort(y_pred_proba)[::-1]
  y_true_top_k = y_true[indices][:k]
  y_score_top_k = y_pred_proba[indices][:k]

  # Calculate average precision
  ap = average_precision_score(y_true_top_k, y_score_top_k)

  return ap

In [ ]:
def calculate_ndcg(y_true, y_pred_proba, k=20):
  """Calculates NDCG for ranking tasks.

  Args:
    y_true: True relevance scores.
    y_score: Predicted relevance scores.
    k: Number of top items to consider.

  Returns:
    NDCG value.
  """

  # Calculate NDCG
  ndcg = ndcg_score(y_true, y_pred_proba, k=k)

  return ndcg

In [ ]:
def calculate_ndcg_at_k(true_labels, predicted_scores, k):
    """Calculates Normalized Discounted Cumulative Gain (NDCG) at k.

    Args:
        true_labels: A list or NumPy array of true labels.
        predicted_scores: A list or NumPy array of predicted scores.
        k: The number of top predictions to consider.

    Returns:
        The NDCG at k.
    """

    if len(true_labels) == 0:
        return 0

    # Ensure k does not exceed the number of predicted scores
    k = min(k, len(predicted_scores))

    def dcg(relevance, k):
        relevance = np.array(relevance)[:k]  # Take top k relevant items
        return np.sum((2 ** relevance - 1) / np.log2(np.arange(2, len(relevance) + 2)))

    # Sort indices by descending predicted scores
    indices = np.argsort(predicted_scores)[::-1][:k]

    # Get top k true labels and calculate DCG
    top_k_true_labels = true_labels[indices]
    dcg_k = dcg(top_k_true_labels, k)

    # Calculate Ideal Discounted Cumulative Gain (IDCG)
    ideal_labels = np.sort(true_labels)[::-1][:k]
    idcg_k = dcg(ideal_labels, k)

    # Calculate NDCG
    ndcg_k = dcg_k / idcg_k if idcg_k > 0 else 0

    return ndcg_k

In [ ]:
def compute_and_display_confusion_matrix(y_true, y_pred_proba, group=None):
    """
    Compute and display the confusion matrix for a given target group.

    Parameters:
    - y_true: True labels for the test set or validation fold.
    - y_pred_proba: Predicted probability scores from the model.
    - group: The target group for labeling purposes.
    
    Returns:
    - None
    """
    y_pred = (y_pred_proba > 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    # Display confusion matrix as a heatmap
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False, 
                xticklabels=['Predicted Negative', 'Predicted Positive'], 
                yticklabels=['Actual Negative', 'Actual Positive'])
    plt.title(f'Confusion Matrix for {group}' if group else 'Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
def calculate_precision_score(y_true, y_pred_proba):
    """Calculates precision score for binary classification.

    Args:
        y_true: True binary labels.
        y_pred_proba: Predicted probability scores.
        average: Method for averaging precision across classes.
        pos_label: Positive class label.

    Returns:
        Precision score.
    """

    y_pred = (y_pred_proba > 0.5).astype(int)
    precision = precision_score(y_true, y_pred)
    return precision

In [ ]:
def plot_roc_curve(y_true, y_pred_proba, group=None):
    """Plots the ROC curve for binary classification for each target group.

    Args:
        y_true: True binary labels.
        y_pred_proba: Predicted probability scores.
        group: The target group for labeling purposes.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba, pos_label=1)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve for {group}' if group else 'Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
# Function to evaluate model on validation data for each target group and return metrics in a DataFrame
def evaluate_model_by_group(X_val, y_val, target_group_val, model, k=20):
    """Evaluates the model on validation data, calculates various metrics for each target group, and returns a DataFrame."""
    
    # Get predicted probabilities for the positive class
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Create a DataFrame with predicted probabilities, true labels, and target groups
    results_df = pd.DataFrame({
        'y_true': y_val.values.ravel(),
        'y_pred_proba': y_pred_proba,
        'target_group': target_group_val.values.ravel()
    })

    # Prepare a list to hold the metrics for each group
    metrics_list = []

    # Calculate precision@k and other metrics for each target group
    for group in results_df['target_group'].unique():
        group_df = results_df[results_df['target_group'] == group]
        
        # Calculate precision@k, average precision@k, ndcg@k, and other metrics
        precision_at_k = calculate_precision_at_k(group_df['y_true'].values, group_df['y_pred_proba'].values, k=k)
        average_precision_at_k = average_precision_top_k(group_df['y_true'].values, group_df['y_pred_proba'].values, k=k)
        ndcg_at_k = calculate_ndcg_at_k(group_df['y_true'].values, group_df['y_pred_proba'].values, k=k)
        precision_score_value = precision_score(group_df['y_true'].values, (group_df['y_pred_proba'].values > 0.5).astype(int),
                                               zero_division=0)

        # Plot ROC curve and confusion matrix for the group
        plot_roc_curve(group_df['y_true'].values, group_df['y_pred_proba'].values, group)
        compute_and_display_confusion_matrix(group_df['y_true'].values, group_df['y_pred_proba'].values, group)

        # Store the results in a dictionary
        metrics_dict = {
            'target_group': group,
            'precision_at_k': precision_at_k,
            'average_precision_at_k': average_precision_at_k,
            'ndcg_at_k': ndcg_at_k,
            'precision_score': precision_score_value
        }
        
        # Append the metrics for this group to the list
        metrics_list.append(metrics_dict)

    # Convert the list of metrics dictionaries to a DataFrame
    metrics_df = pd.DataFrame(metrics_list)
    
    return metrics_df, results_df


## Random Forest Baseline

In [ ]:
def rf_baseline_model(X_train, y_train):
  """Trains a baseline model using K-fold cross-validation and returns the model and average precision score.

  Args:
      X_train: Training features (assumes pandas DataFrame format).
      y_train: Training labels.
      k_folds: Number of folds for cross-validation (default: 5).
      random_state: Random state for reproducibility (default: 42).
      model_class: Model class to use (e.g., RandomForestClassifier) (default: RandomForestClassifier).

  Returns:
      A tuple containing the trained baseline model and the average precision score.
  """

  kf = KFold(n_splits=5, shuffle=True, random_state=42)

  baseline_model = RandomForestClassifier(random_state=42, n_jobs=-1)
  baseline_scores = []

  for train_index, val_index in kf.split(X_train):
    # Get row-wise indexing based on train/val indices
    X_train_fold = X_train.iloc[train_index]
    X_val_fold = X_train.iloc[val_index]
    y_train_fold = y_train.iloc[train_index]
    y_val_fold = y_train.iloc[val_index]

    baseline_model.fit(X_train_fold, y_train_fold.values.ravel())

    # Get probabilities for the positive class only (assuming binary classification)
    y_pred_proba = baseline_model.predict_proba(X_val_fold)[:, 1]

    # Apply the 0.5 threshold to the positive class probability
    y_pred = (y_pred_proba > 0.5).astype(int)

    # Compute precision for the validation set
    score = precision_score(y_val_fold, y_pred)
    baseline_scores.append(score)

  # Calculate and return the average precision
  precision = np.mean(baseline_scores)
  precision_std = np.std(baseline_scores)
  return baseline_model, precision, precision_std

### Baseline for Roberta:

In [ ]:
group_metrics_baseline_roberta, result_baseline_roberta = evaluate_model_by_group(X_val_roberta, y_val, target_group_val_split,
                                                         baseline_model_roberta, k=20)

In [ ]:
group_metrics_baseline_roberta_t, result_baseline_roberta_t = evaluate_model_by_group(X_train_roberta_train, y_train_train, target_group_train_split,
                                                         baseline_model_roberta, k=20)

In [ ]:
print(group_metrics_baseline_roberta_t['precision_score'].mean())
print(group_metrics_baseline_roberta['precision_score'].mean())

### Baseline on distilroberta

In [ ]:
group_metrics_baseline_distilroberta, result_baseline_distilroberta = evaluate_model_by_group(X_val_distilroberta, y_val, target_group_val_split, 
                                                               baseline_model_distilroberta, k=20)

In [ ]:
group_metrics_baseline_distilroberta_t, result_baseline_distilroberta_t = evaluate_model_by_group(X_train_distilroberta_train, y_train_train, target_group_train_split, 
                                                               baseline_model_distilroberta, k=20)

In [ ]:
print(group_metrics_baseline_distilroberta_t['precision_score'].mean()) #train
print(group_metrics_baseline_distilroberta['precision_score'].mean()) #val

### Baseline on Distrilroberta Mitch

In [ ]:
group_metrics_baseline_distilroberta_mitch, result_baseline_distilroberta_mitch = evaluate_model_by_group(X_val_distilroberta_mitch, y_val,
                                                                     target_group_val_split,
                                                                     baseline_model_distilroberta_mitch, k=20)

In [ ]:
group_metrics_baseline_distilroberta_mitch_t, result_baseline_distilroberta_mitch_t = evaluate_model_by_group(X_train_distilroberta_mitch_train, y_train_train,
                                                                     target_group_train_split,
                                                                     baseline_model_distilroberta_mitch, k=20)

In [ ]:
print(group_metrics_baseline_distilroberta_mitch_t['precision_score'].mean())
print(group_metrics_baseline_distilroberta_mitch['precision_score'].mean())

**The model seems to be overfitting on for all target groups of the data in all the three dataset.**

**In line to remedy the problem, we will manually experiment hypeparameters such as n-estimator, maximum depth and dimentionalty size using PCA**.

## Experiment on Maximum Depth = 5 for Roberta 

In [ ]:
rf_max_depth = RandomForestClassifier(max_depth=5, random_state=42)
scores = cross_val_score(rf_max_depth, X_train_roberta_train, y_train_train.values.ravel(), cv=5)
rf_max_depth.fit(X_train_roberta_train, y_train_train.values.ravel())
print("Max Depth 5: ", scores.mean())

In [ ]:
group_metrics_robertaB, result_robertaB = evaluate_model_by_group(X_val_roberta, y_val, target_group_val_split,
                                                         rf_max_depth, k=20)

In [ ]:
group_metrics_roberta_tB, result_roberta_tB = evaluate_model_by_group(X_train_roberta_train, y_train_train, target_group_train_split,
                                                         rf_max_depth, k=20)

In [ ]:
print(group_metrics_roberta_tB['precision_score'].mean())
print(group_metrics_robertaB['precision_score'].mean())

* This reduced overfitting significantly in roberta data

## Experiment on number of estimator - 1000 and 500

In [ ]:
rf_n_estimator = RandomForestClassifier(n_estimators=1000, random_state=42)
scores = cross_val_score(rf_n_estimator, X_train_roberta_train, y_train_train.values.ravel(), cv=5)
rf_n_estimator.fit(X_train_roberta_train, y_train_train.values.ravel())
print("n_estimator 1000: ", scores.mean())

In [ ]:
group_metrics_robertaC, result_robertaC = evaluate_model_by_group(X_val_roberta, y_val, target_group_val_split,
                                                         rf_n_estimator, k=20)

In [ ]:
group_metrics_roberta_tC, result_roberta_tC = evaluate_model_by_group(X_train_roberta_train, y_train_train, target_group_train_split,
                                                         rf_n_estimator, k=20)

In [ ]:
print(group_metrics_roberta_tC['precision_score'].mean())
print(group_metrics_robertaC['precision_score'].mean())

* Overfitted

## Third Model with Principal Component Analysis's n_component = 10 and random forest's max depth =5

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)  # Here I reduced the features to 10 principal components
X_train_pca = pca.fit_transform(X_train_roberta_train)
X_val_pca = pca.fit_transform(X_val_roberta)


# Train the Random Forest on the PCA-transformed data
rf_depth_pca = RandomForestClassifier(max_depth=5, random_state=42)
scores = cross_val_score(rf_depth_pca, X_train_pca, y_train_train.values.ravel(), cv=5)
rf_depth_pca.fit(X_train_pca, y_train_train.values.ravel())
print("Max Depth 5 on PCA data: ", scores.mean())


In [ ]:
group_metrics_robertaD, result_robertaD = evaluate_model_by_group(X_val_pca, y_val, target_group_val_split,
                                                         rf_depth_pca, k=20)

In [ ]:
group_metrics_baseline_roberta_tD, result_baseline_roberta_tD = evaluate_model_by_group(X_train_pca, y_train_train, target_group_train_split,
                                                         rf_depth_pca, k=20)

In [ ]:
print(group_metrics_baseline_roberta_tD['precision_score'].mean()) #train
print(group_metrics_robertaD['precision_score'].mean()) #val

## Model - Bagging

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Initialize a Decision Tree as the base model
base_model = DecisionTreeClassifier(random_state=42)
# Bagging with Decision Tree
bagging_model = BaggingClassifier(base_estimator=base_model, n_estimators=100, random_state=42, oob_score=True)
scores = cross_val_score(bagging_model, X_train_pca, y_train_train.values.ravel(), cv=5)
bagging_model.fit(X_train_pca, y_train_train.values.ravel())
print("Ensemble on PCA data: ", scores.mean())
# Fit the model
bagging_model.fit(X_train_pca, y_train_train)

In [ ]:
group_metrics_robertaE, result_baseline_robertaE = evaluate_model_by_group(X_val_pca, y_val, target_group_val_split,
                                                         bagging_model, k=20)

In [ ]:
group_metrics_roberta_tE, result_baseline_roberta_tE = evaluate_model_by_group(X_train_pca, y_train_train, target_group_train_split,
                                                         bagging_model, k=20)

In [ ]:
print(group_metrics_roberta_tE['precision_score'].mean())
print(group_metrics_robertaE['precision_score'].mean())

* Overfitted

## Hyperparameter Tuning to optimise model

In [ ]:

from sklearn.pipeline import Pipeline

# Create a pipeline with PCA and Random Forest
pipeline = Pipeline([
    ('pca', PCA()),  # PCA step
    ('rf', RandomForestClassifier(random_state=42))  # Random Forest step
])

# Define the parameter grid including PCA n_components and Random Forest hyperparameters
param_grid = {
    'pca__n_components': [5, 10, 20],  # Tuning the number of principal components
    'rf__max_depth': [5, 15, 30],      # Tuning Random Forest hyperparameters
    'rf__min_samples_split': [5, 25, 50],
    'rf__min_samples_leaf': [2, 7, 12],
    'rf__n_estimators': [5, 50, 100, 200]
}

# Use GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='precision', n_jobs=-1)

# Fit the model on the training data
grid_search.fit(X_train_roberta_train, y_train_train.values.ravel())

# Print the score for each combination of hyperparameters
print("Grid search results for each combination of hyperparameters:")

for mean_score, params in zip(grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['params']):
    print(f"Parameters: {params}, Mean CV Precision: {mean_score:.4f}")

# Print the best parameters and best score
print("\nBest parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

best_model = grid_search.best_estimator_

In [ ]:
group_metrics_baseline_robertaF, result_baseline_robertaF = evaluate_model_by_group(X_val_roberta, y_val, target_group_val_split,
                                                         best_model, k=20)

In [ ]:
group_metrics_baseline_robertatF, result_baseline_robertatF = evaluate_model_by_group(X_train_roberta_train, y_train_train, target_group_train_split,
                                                         best_model, k=20)

In [ ]:
print(group_metrics_baseline_robertatF['precision_score'].mean()) #train
print(group_metrics_baseline_robertaF['precision_score'].mean())# val

# This model still overfitted.

## Roberta Score Summary

In [ ]:
print('Baseline Model on Train:',group_metrics_baseline_roberta_t['precision_score'].mean())
print('Baseline Model on Val:',group_metrics_baseline_roberta['precision_score'].mean())
print('---------------------------------------')
print('Max depth =5 Model on Train:',group_metrics_roberta_tB['precision_score'].mean())
print('Max depth =5 Model on Val:', group_metrics_robertaB['precision_score'].mean())
print('---------------------------------------')
print('n_estimator = 1000 Model on Train:',group_metrics_roberta_tC['precision_score'].mean())
print('n_estimator = 1000 Model on Val:', group_metrics_robertaC['precision_score'].mean())
print('---------------------------------------')
print('PCA(n_component = 10) and max depth =5 Model on Train:',group_metrics_baseline_roberta_tD['precision_score'].mean())
print('PCA(n_component = 10) and max depth =5 Model on Val:', group_metrics_robertaD['precision_score'].mean())
print('---------------------------------------')
print('Bagging Model on Train:',group_metrics_roberta_tE['precision_score'].mean())
print('Bagging Model on Val:', group_metrics_robertaE['precision_score'].mean())
print('---------------------------------------')
print('Hyperparameter Tuning Best Model on Train:',group_metrics_baseline_robertatF['precision_score'].mean())
print('Hyperparameter Tuning Best Model on Val:', group_metrics_baseline_robertaF['precision_score'].mean())

**Final model selected for roberta is PCA(n_component = 10) and max depth = 5)**

## Evaluate the final model selected for roberta on its test set

In [ ]:
#PCA and Max Depth
X_test_pca = pca.fit_transform(X_test_roberta)

In [ ]:
group_metrics_roberta_test, result_roberta_test = evaluate_model_by_group(X_test_pca, y_test, target_group_test,
                                                         rf_depth_pca, k=20)

## Distilroberta Data

### Max Depth = 5 experiment on Distilroberta

In [ ]:
rf_max_depth = RandomForestClassifier(max_depth=5, random_state=42)
scores = cross_val_score(rf_max_depth, X_train_distilroberta_train, y_train_train.values.ravel(), cv=5)
rf_max_depth.fit(X_train_distilroberta_train, y_train_train.values.ravel())
print("Max Depth 5: ", scores.mean())

In [ ]:
# evaluate on validation set
group_metrics_distilrobertaB, result_distilrobertaB = evaluate_model_by_group(X_val_distilroberta, y_val, target_group_val_split,
                                                         rf_max_depth, k=20)

In [ ]:
# evaluate on training set
group_metrics_distilroberta_tB, result_distilroberta_tB = evaluate_model_by_group(X_train_distilroberta_train, y_train_train, target_group_train_split,
                                                         rf_max_depth, k=20)

In [ ]:
print(group_metrics_distilroberta_tB['precision_score'].mean()) #train
print(group_metrics_distilrobertaB['precision_score'].mean()) #val

### 1000 n-estimator experiment on Distilroberta

In [ ]:
rf_n_estimator = RandomForestClassifier(n_estimators=1000, random_state=42)
scores = cross_val_score(rf_n_estimator, X_train_distilroberta_train, y_train_train.values.ravel(), cv=5)
rf_n_estimator.fit(X_train_distilroberta_train, y_train_train.values.ravel())


In [ ]:
# evaluate on target group
group_metrics_distilrobertaC, result_distilrobertaC = evaluate_model_by_group(X_val_distilroberta, y_val, target_group_val_split,
                                                         rf_n_estimator, k=20)

In [ ]:
group_metrics_distilroberta_tC, result_distilroberta_tC = evaluate_model_by_group(X_train_distilroberta_train, y_train_train, target_group_train_split,
                                                         rf_n_estimator, k=20)

In [ ]:
print(group_metrics_distilroberta_tC['precision_score'].mean())
print(group_metrics_distilrobertaC['precision_score'].mean())

# this overfitted. It got all the taining correct but score 0.6 in validation set

### PCA(n_components=10) and Max depth = 5

In [ ]:
pca = PCA(n_components=10) 
X_distiltrain_pca = pca.fit_transform(X_train_distilroberta_train)
X_distilval_pca = pca.fit_transform(X_val_distilroberta)

rf_depth_pca = RandomForestClassifier(max_depth=5, random_state=42)
scores = cross_val_score(rf_depth_pca, X_distiltrain_pca, y_train_train.values.ravel(), cv=5)
rf_depth_pca.fit(X_distiltrain_pca, y_train_train.values.ravel())
print("Max Depth 5 on PCA data: ", scores.mean())

In [ ]:
group_metrics_distilrobertaD, result_distilrobertaD = evaluate_model_by_group(X_distilval_pca, y_val, target_group_val_split,
                                                         rf_depth_pca, k=20)

In [ ]:
group_metrics_distilroberta_tD, result_distilroberta_tD = evaluate_model_by_group(X_distiltrain_pca, y_train_train, target_group_train_split,
                                                         rf_depth_pca, k=20)

In [ ]:
print(group_metrics_distilroberta_tD['precision_score'].mean())
print(group_metrics_distilrobertaD['precision_score'].mean())

### Bagging

In [ ]:
# Initialize a Decision Tree as the base model
base_model = DecisionTreeClassifier(random_state=42)
# Bagging with Decision Tree
bagging_model = BaggingClassifier(base_estimator=base_model, n_estimators=100, random_state=42, oob_score=True)
scores = cross_val_score(bagging_model, X_distiltrain_pca, y_train_train.values.ravel(), cv=5)
bagging_model.fit(X_distiltrain_pca, y_train_train.values.ravel())
print("Ensemble on PCA data: ", scores.mean())
# Fit the model
bagging_model.fit(X_distiltrain_pca, y_train_train)

In [ ]:
group_metrics_distilrobertaE, result_distilrobertaE = evaluate_model_by_group(X_distilval_pca, y_val, target_group_val_split,
                                                         bagging_model, k=20)

In [ ]:
group_metrics_distilroberta_tE, result_distilroberta_tE = evaluate_model_by_group(X_distiltrain_pca, y_train_train, target_group_train_split,
                                                         bagging_model, k=20)

In [ ]:
print(group_metrics_distilroberta_tE['precision_score'].mean())
print(group_metrics_distilrobertaE['precision_score'].mean())

#Overfitted

### Hyperparameter Tuning

In [ ]:
# Create a pipeline with PCA and Random Forest
pipeline = Pipeline([
    ('pca', PCA()),  # PCA step
    ('rf', RandomForestClassifier(random_state=42))  # Random Forest step
])

# Define the parameter grid including PCA n_components and Random Forest hyperparameters
param_grid = {
    'pca__n_components': [5, 10, 20],  # Tuning the number of principal components
    'rf__max_depth': [5, 15, 30],      # Tuning Random Forest hyperparameters
    'rf__min_samples_split': [5, 25, 50],
    'rf__min_samples_leaf': [2, 7, 12],
    'rf__n_estimators': [5, 50, 100, 200]
}

# Use GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='precision', n_jobs=-1)

# Fit the model on the training data
grid_search.fit(X_train_distilroberta_train, y_train_train.values.ravel())

# Print the score for each combination of hyperparameters
print("Grid search results for each combination of hyperparameters:")

for mean_score, params in zip(grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['params']):
    print(f"Parameters: {params}, Mean CV Precision: {mean_score:.4f}")

# Print the best parameters and best score
print("\nBest parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

best_model = grid_search.best_estimator_

In [ ]:
group_metrics_distilrobertaF, result_distilrobertaF = evaluate_model_by_group(X_val_distilroberta, y_val, target_group_val_split,
                                                         best_model, k=20)

In [ ]:
group_metrics_distilrobertatF, result_distilrobertatF = evaluate_model_by_group(X_train_distilroberta_train, y_train_train, target_group_train_split,
                                                         best_model, k=20)

In [ ]:
print(group_metrics_distilrobertatF['precision_score'].mean())
print(group_metrics_distilrobertaF['precision_score'].mean())

# overfitted

In [ ]:
## Summary of Distilroberta Scores

In [ ]:
print('Hyperparameter Tuning on Training:',group_metrics_distilrobertatF['precision_score'].mean())
print('Hyperparameter Tuning on Validation:',group_metrics_distilrobertaF['precision_score'].mean())
print('-----------------------------------------------------')
print('Bagging on Training:',group_metrics_distilroberta_tE['precision_score'].mean())
print('Bagging on Validation:',group_metrics_distilrobertaE['precision_score'].mean())
print('-----------------------------------------------------')
print('PCA-10, Max depth -5 on training',group_metrics_distilroberta_tD['precision_score'].mean())
print('PCA-10, Max depth -5 on validation',group_metrics_distilrobertaD['precision_score'].mean())
print('-----------------------------------------------------')
print('1000 estimator on training',group_metrics_distilroberta_tC['precision_score'].mean())
print('1000 estimator on validation',group_metrics_distilrobertaC['precision_score'].mean())
print('-----------------------------------------------------')
print('Max depth - 5 on training',group_metrics_distilroberta_tB['precision_score'].mean())
print('Max depth - 5 on validation',group_metrics_distilrobertaB['precision_score'].mean())
print('-----------------------------------------------------')
print('Baseline Model on training',group_metrics_baseline_distilroberta_t['precision_score'].mean())
print('Baseline Model on validation',group_metrics_baseline_distilroberta['precision_score'].mean())

**PCA-10, Max depth -5 was selected because it generalised better than the others**

In [ ]:
X_test_pca = pca.fit_transform(X_test_distilroberta)
group_metrics_distilroberta_test, result_distilroberta_test = evaluate_model_by_group(X_test_pca, y_test, target_group_test,
                                                         rf_depth_pca, k=20)

## Distilroberta_mitch

### max_depth Model

In [ ]:
rf_max_depth = RandomForestClassifier(max_depth=5, random_state=42)
scores = cross_val_score(rf_max_depth, X_train_distilroberta_mitch_train, y_train_train.values.ravel(), cv=5)
rf_max_depth.fit(X_train_distilroberta_mitch_train, y_train_train.values.ravel())
print("Max Depth 5: ", scores.mean())

In [ ]:
group_metrics_distilroberta_mitchB, result_distilroberta_mitchB = evaluate_model_by_group(X_val_distilroberta_mitch, y_val, target_group_val_split,
                                                         rf_max_depth, k=20)

In [ ]:
group_metrics_distilroberta_mitch_tB, result_distilroberta_mitch_tB = evaluate_model_by_group(X_train_distilroberta_mitch_train, y_train_train, target_group_train_split,
                                                         rf_max_depth, k=20)

### n_estimator model

In [ ]:
rf_n_estimator = RandomForestClassifier(n_estimators=1000, random_state=42)
scores = cross_val_score(rf_n_estimator, X_train_distilroberta_mitch_train, y_train_train.values.ravel(), cv=5)
rf_n_estimator.fit(X_train_distilroberta_mitch_train, y_train_train.values.ravel())
print("n_estimator 1000: ", scores.mean())

group_metrics_distilroberta_mitchC, result_distilroberta_mitchC = evaluate_model_by_group(X_val_distilroberta_mitch, y_val, target_group_val_split,
                                                         rf_n_estimator, k=20)


In [ ]:
group_metrics_distilroberta_mitch_tC, result_distilroberta_mitch_tC = evaluate_model_by_group(X_train_distilroberta_mitch_train, y_train_train, target_group_train_split,
                                                         rf_n_estimator, k=20)

### pca=10, max depth 5 model

In [ ]:
pca = PCA(n_components=10)
X_distiltrain_mitch_pca = pca.fit_transform(X_train_distilroberta_mitch_train)
X_distilval_mitch_pca = pca.fit_transform(X_val_distilroberta_mitch)

#Train the Random Forest on the PCA-transformed data
rf_depth_pca = RandomForestClassifier(max_depth=5, random_state=42)
scores = cross_val_score(rf_depth_pca, X_distiltrain_mitch_pca, y_train_train.values.ravel(), cv=5)
rf_depth_pca.fit(X_distiltrain_mitch_pca, y_train_train.values.ravel())
print("Max Depth 5 on PCA data: ", scores.mean())

In [ ]:
group_metrics_distilroberta_mitchD, result_distilroberta_mitchD = evaluate_model_by_group(X_distilval_mitch_pca, y_val, target_group_val_split,
                                                         rf_depth_pca, k=20)

In [ ]:
group_metrics_distilroberta_mitch_tD, result_distilroberta_mitch_tD = evaluate_model_by_group(X_distiltrain_mitch_pca, y_train_train, target_group_train_split,
                                                         rf_depth_pca, k=20)

### Bagging model

In [ ]:
# Initialize a Decision Tree as the base model
base_model = DecisionTreeClassifier(random_state=42)
# Bagging with Decision Tree
bagging_model = BaggingClassifier(base_estimator=base_model, n_estimators=100, random_state=42, oob_score=True)
scores = cross_val_score(bagging_model, X_distiltrain_mitch_pca, y_train_train.values.ravel(), cv=5)
bagging_model.fit(X_distiltrain_mitch_pca, y_train_train.values.ravel())
print("Ensemble on PCA data: ", scores.mean())
# Fit the model
bagging_model.fit(X_distiltrain_mitch_pca, y_train_train)



In [ ]:
group_metrics_distilroberta_mitchE, result_distilroberta_mitchE = evaluate_model_by_group(X_distilval_mitch_pca, y_val, target_group_val_split,
                                                         bagging_model, k=20)


In [ ]:

group_metrics_distilroberta_mitch_tE, result_distilroberta_mitch_tE = evaluate_model_by_group(X_distiltrain_mitch_pca, y_train_train, target_group_train_split,
                                                         bagging_model, k=20)

### Hyperparameter tuning

In [ ]:

#Create a pipeline with PCA and Random Forest
pipeline = Pipeline([
    ('pca', PCA()), 
    ('rf', RandomForestClassifier(random_state=42))  
])

#Define the parameter grid including PCA n_components and Random Forest hyperparameters
param_grid = {
    'pca__n_components': [5, 10, 20],  
    'rf__max_depth': [5, 15, 30],     
    'rf__min_samples_split': [5, 25, 50],
    'rf__min_samples_leaf': [2, 7, 12],
    'rf__n_estimators': [5, 50, 100, 200]
}

# Use GridSearchCV to perform hyperparameter tuning
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='precision', n_jobs=-1)

# Fit the model on the training data
grid_search.fit(X_train_distilroberta_mitch_train, y_train_train.values.ravel())

# Print the score for each combination of hyperparameters
print("Grid search results for each combination of hyperparameters:")

for mean_score, params in zip(grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['params']):
    print(f"Parameters: {params}, Mean CV Precision: {mean_score:.4f}")

# Print the best parameters and best score
print("\nBest parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

best_model = grid_search.best_estimator_

In [ ]:
group_metrics_distilroberta_mitchF, result_distilroberta_mitchF = evaluate_model_by_group(X_val_distilroberta_mitch, y_val, target_group_val_split,
                                                         best_model, k=20)

In [ ]:
group_metrics_distilroberta_mitchtF, result_distilrobertat_mitchF = evaluate_model_by_group(X_train_distilroberta_mitch_train, y_train_train, target_group_train_split,
                                                         best_model, k=20)

### Distilroberta Scores

In [ ]:
print('Hyperparameter tuning on training data',group_metrics_distilroberta_mitchtF['precision_score'].mean())
print('Hyperparameter tuning on validation data',group_metrics_distilroberta_mitchF['precision_score'].mean())
print('-----------------------------------------------------')
print('Bagging on training data',group_metrics_distilroberta_mitch_tE['precision_score'].mean())
print('Bagging on validation data',group_metrics_distilroberta_mitchE['precision_score'].mean())
print('-----------------------------------------------------')
print('PCA and max depth model on training data',group_metrics_distilroberta_mitch_tD['precision_score'].mean())
print('PCA and max depth model on validation data',group_metrics_distilroberta_mitchD['precision_score'].mean())
print('-----------------------------------------------------')
print('n_estimator on training data',group_metrics_distilroberta_mitch_tC['precision_score'].mean())
print('n_estimator on validation data',group_metrics_distilroberta_mitchC['precision_score'].mean())
print('-----------------------------------------------------')
print('max depth on training data',group_metrics_distilroberta_mitch_tB['precision_score'].mean())
print('max depth on validation data',group_metrics_distilrobertaB['precision_score'].mean())
print('-----------------------------------------------------')
print('baseline model on training data',group_metrics_baseline_distilroberta_mitch_t['precision_score'].mean())
print('baseline model on validation data',group_metrics_baseline_distilroberta_mitch['precision_score'].mean())

In [ ]:
X_test_pca = pca.fit_transform(X_test_distilroberta_mitch)
group_metrics_distilroberta_mitch_test, result_distilroberta_mitch_test = evaluate_model_by_group(X_test_pca, y_test, target_group_test,
                                                         rf_depth_pca, k=20)

In [ ]:
group_metrics_roberta_test.columns

In [ ]:
merged_df_1_2 = pd.merge(group_metrics_roberta_test, group_metrics_distilroberta_test, on='target_group', how='inner')
final_merged_df = pd.merge(merged_df_1_2, group_metrics_distilroberta_mitch_test, on='target_group', how='inner')
sorted_metrics = final_merged_df.sort_values(by='target_group', ascending=True)

In [ ]:
sorted_metrics.to_excel('rfc_test_result.xlsx', index=False)